# Phase 1 — Baseline Model for ADME Prediction

In this notebook we:
1. Load the **Lipophilicity** dataset from Therapeutics Data Commons
2. Explore the data and understand SMILES notation
3. Convert molecules to **Morgan fingerprints** using RDKit
4. Train a **Random Forest** baseline and evaluate it

**Goal:** Establish a solid baseline to beat later with ChemBERTa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load the Dataset

We use the **Lipophilicity (AstraZeneca)** dataset from TDC. Lipophilicity (logD) 
measures how well a drug dissolves in fats vs water — a key factor in how drugs 
are absorbed and distributed in the body.

In [ ]:
from tdc.single_pred import ADME

data = ADME(name='Lipophilicity_AstraZeneca')
df = data.get_data()
print(f'Dataset size: {len(df)} compounds')
df.head()

### What's in the data?

- **Drug**: The molecule as a SMILES string
- **Drug_ID**: ChEMBL identifier
- **Y**: The target — experimental logD value (lipophilicity)

In [ ]:
print(df.describe())
print(f'\nSMILES example: {df["Drug"].iloc[0]}')

## 2. Explore the Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
axes[0].hist(df['Y'], bins=50, color='#2563eb', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('logD (Lipophilicity)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Lipophilicity Values')
axes[0].axvline(df['Y'].mean(), color='red', linestyle='--', label=f'Mean = {df["Y"].mean():.2f}')
axes[0].legend()

# SMILES length distribution
df['smiles_len'] = df['Drug'].str.len()
axes[1].hist(df['smiles_len'], bins=50, color='#16a34a', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('SMILES String Length')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Molecule Sizes')

plt.tight_layout()
plt.savefig('../assets/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualize some molecules

Let's see what these SMILES strings actually look like as 2D structures.

In [ ]:
# Pick 8 random molecules to visualize
sample = df.sample(8, random_state=42)
mols = [Chem.MolFromSmiles(s) for s in sample['Drug']]
legends = [f'logD = {y:.2f}' for y in sample['Y']]

img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 300), legends=legends)
img

## 3. Featurize: SMILES → Morgan Fingerprints

Morgan fingerprints encode the local chemical environment around each atom 
in a fixed-length bit vector. Think of it as a molecular "hash" that captures 
structural patterns.

In [ ]:
def smiles_to_morgan(smiles, radius=2, n_bits=2048):
    """Convert SMILES to Morgan fingerprint bit vector."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    return np.array(fp)


# Test on aspirin
aspirin_fp = smiles_to_morgan('CC(=O)OC1=CC=CC=C1C(=O)O')
print(f'Fingerprint shape: {aspirin_fp.shape}')
print(f'Bits set: {aspirin_fp.sum()} / {len(aspirin_fp)} ({100*aspirin_fp.mean():.1f}%)')

In [ ]:
# Featurize entire dataset
print('Generating fingerprints...')
X = np.vstack([smiles_to_morgan(s) for s in df['Drug']])
y = df['Y'].values

print(f'Feature matrix: {X.shape}')
print(f'Target vector:  {y.shape}')

## 4. Split the Data

We use TDC's **scaffold split**, which groups molecules by their core structure.
This is more realistic than random splitting because in real drug discovery,
you test on novel scaffolds, not slight variations of training molecules.

In [ ]:
split = data.get_split(method='scaffold', seed=42)

train_df = split['train']
valid_df = split['valid']
test_df  = split['test']

print(f'Train: {len(train_df)}, Valid: {len(valid_df)}, Test: {len(test_df)}')

# Featurize each split
X_train = np.vstack([smiles_to_morgan(s) for s in train_df['Drug']])
X_valid = np.vstack([smiles_to_morgan(s) for s in valid_df['Drug']])
X_test  = np.vstack([smiles_to_morgan(s) for s in test_df['Drug']])

y_train = train_df['Y'].values
y_valid = valid_df['Y'].values
y_test  = test_df['Y'].values

## 5. Train Random Forest Baseline

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)

rf.fit(X_train, y_train)
print('Training complete!')

In [ ]:
def eval_and_print(model, X, y, name):
    """Evaluate model and print metrics."""
    y_pred = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    print(f'{name:>12s}  |  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}  |  R²: {r2:.4f}')
    return y_pred

print(f'{"Set":>12s}  |  {"RMSE":>12s}  |  {"MAE":>11s}  |  {"R²":>9s}')
print('-' * 62)
_ = eval_and_print(rf, X_train, y_train, 'Train')
_ = eval_and_print(rf, X_valid, y_valid, 'Valid')
y_pred_test = eval_and_print(rf, X_test, y_test, 'Test')

In [ ]:
# Cross-validation
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
print(f'5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 6. Visualize Results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test, y_pred_test, alpha=0.4, s=20, c='#2563eb', edgecolors='none')

lims = [min(y_test.min(), y_pred_test.min()) - 0.5,
        max(y_test.max(), y_pred_test.max()) + 0.5]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='Perfect prediction')

rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)
ax.text(0.05, 0.95, f'R² = {r2:.3f}\nRMSE = {rmse:.3f}',
        transform=ax.transAxes, fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_xlabel('Actual logD', fontsize=13)
ax.set_ylabel('Predicted logD', fontsize=13)
ax.set_title('Random Forest Baseline — Lipophilicity (Test Set)', fontsize=14)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../assets/rf_baseline_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Error Analysis

Let's look at where the model struggles most.

In [ ]:
errors = np.abs(y_test - y_pred_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution
axes[0].hist(errors, bins=40, color='#dc2626', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Absolute Error')
axes[0].set_ylabel('Count')
axes[0].set_title('Error Distribution')
axes[0].axvline(errors.mean(), color='black', linestyle='--',
                label=f'Mean = {errors.mean():.3f}')
axes[0].legend()

# Error vs actual value
axes[1].scatter(y_test, errors, alpha=0.3, s=15, c='#dc2626')
axes[1].set_xlabel('Actual logD')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Error vs Actual Value')

plt.tight_layout()
plt.savefig('../assets/rf_baseline_errors.png', dpi=150, bbox_inches='tight')
plt.show()

## Next Steps

This baseline gives us a benchmark to improve on. In the next notebooks:

- **Phase 2**: Try different molecular representations (MACCS keys, RDKit descriptors, combined features)
- **Phase 3**: Fine-tune ChemBERTa to learn representations directly from SMILES strings

The key question: Can a language model that *reads* SMILES strings outperform 
hand-crafted molecular fingerprints?